# 리포트 55 — 한 순간의 (R_b, f_d) 는 랭크 2 이고, 수신기를 하나 더하면 위치가 풀린다

> ### 한 일
> **송수신 한 쌍이 한 순간에 담는 정보량을 Fisher 정보행렬의 랭크로 세고, 수신기를 더하거나 도래각을 더했을 때 위치 오차가 어디로 가는지를 계산했다.**

### 결과
1. TX–RX 기저선 15.07 m [^1] 형상에서 한 순간의 $(R_b, f_d)$ 는 3차원 위치에 대해 랭크 2 [^2] 를 만든다.
2. 기저선을 축으로 표적을 돌리면 $R_b$ 변화가 최대 1.4e-14 m [^3] 다 — 그 방향의 정보량은 SNR 과 관측시간에 무관하게 0 이다.
3. 수신기를 하나 더 놓으면 랭크 6 [^4] · 위치 RMS 0.19 m [^5] 가 된다 — 그 절대값은 측위 세션 옵션인 PRS 셀(nr100_G3 [^6])에서 푼 값이다.
4. 분해능과 정확도는 다른 양이다 — 5G SSB 는 기준 대역폭 7.20 MHz [^7] 라 셀이 41.64 m [^8] 인데, 같은 반송파에서 PRS 로 가면 3.05 m [^9] 가 된다.
5. 도플러 분해능은 $\Delta f_d = 1/T_{CPI}$ 다 — $T_{CPI}$ 0.032 s [^10] 에서 31.25 Hz [^11].

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 정보량 | 관측이 미지수에 담은 정보량을 Fisher 정보행렬로 세고, 그 랭크로 «무엇이 풀리는가» 를 읽는다 |
| 정확도 하한 | CRLB — 추정 오차가 내려갈 수 있는 이론 하한이다. 대역폭이 분해능을, SNR 이 정확도를 정한다 |
| 거리 규약 | $R_b = c\tau$ 에 계수 2 가 없고 분해능은 $c/B_{ref}$ 다 — 조명원 편과 같은 정의다 |
| 두 열을 가른다 | $\Delta R_b$ 는 선언 규약이고, 거리 빈 $c/f_s$ 는 표본율이 정하는 격자 간격이다 |

### 재현

```bash
cd /home/yunjung/workspace/sionna2
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/verify_observability.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/viz_report04_detector.py
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part09_detector.py
```

| | |
|---|---|
| 출력 | `outputs/verify_observability.json` |
| 소요 | 관측가능성 계산 · 그림은 각각 수 분 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | $\Delta R_b$ 와 잡음대역 규약 |
| [편 51 «수신 → ECA → 거리도플러 → CFAR»](51_chain.ipynb) | 판정이 나는 셀이 무엇인가 |

---

## 분해능과 정확도는 다른 양이다

분해능은 두 표적을 **가르는** 능력이고, 정확도는 한 표적을 **찍는** 정밀도다. 대역폭이 분해능을, SNR 이 정확도를 정한다.

바이스태틱 규약은 $\Delta R_b = c/B_{ref}$ 다($R_b = c\tau$, 계수 2 없이). 아래 표가 그 규약으로 조명원별 셀 크기와 정확도 하한을 나란히 싣는다.

![f6_resolution](../outputs/figures/report04_f6_resolution.png)

**그림 1.** 대역폭이 정하는 것은 분해능인가 정확도인가?

## 조명원별 셀 크기와 CRLB

5G 의 상시 기준신호 SSB 는 기준 대역폭 7.20 MHz [^7] 라 셀이 41.64 m [^8] 다. 같은 반송파에서 PRS 로 가면 98.28 MHz [^12] · 3.05 m [^9] 가 된다.

| 조명원 / 기준신호 | 기준 대역폭 B_ref | ΔR_b = c/B_ref | 거리 빈 c/f_s | σ_Rb (정확도) | σ_fd |
|---|---|---|---|---|---|
| WiFi80 G1 (VHT-LTF) | 76.56 MHz | 3.92 m | 3.75 m | 0.0162 m | 0.128 Hz |
| LTE20 G1 (CRS) | 17.98 MHz | 16.67 m | 9.76 m | 0.0124 m | 0.022 Hz |
| 5G100 G1 (SSB) | 7.20 MHz | 41.64 m | 2.44 m | 1.4901 m | 1.077 Hz |
| 5G100 G3 (PRS) | 98.28 MHz | 3.05 m | 2.44 m | 0.0073 m | 0.073 Hz |

출처 [^13]

오른쪽 두 열은 CRLB 이고, 그 안의 SNR 은 선언값이 아니라 **챔버 동작점에서 잰 SCR** 이다 [^14]. 그래서 두 열은 대역폭만의 함수가 아니라 이 기하·이 표적에 매달린 값이다.

## 두 열을 갈라 읽는다

$\Delta R_b$ 열이 선언 규약이고, 거리 빈은 표본율이 정하는 격자 간격이다. 검출기가 실제로 내는 주엽 폭은 그 닫힌형 대비 비율로 [편 49 «검출기가 실제로 쓰는 커널 그대로 모호함수를…»](49_ambiguity.ipynb) 가 싣는다.

도플러 분해능은 $\Delta f_d = 1/T_{CPI}$ 다 — $T_{CPI}$ 0.032 s [^10] 에서 31.25 Hz [^11]. 그 아래 속도는 [편 52 «탭을 늘리면 환경이 정한 바닥에서 멈추고»](52_eca.ipynb) 의 노치가 먼저 지운다.

## 관측가능성 — 수신기 2대면 위치가 풀린다

검출 판정은 $(R_b, f_d)$ 셀에서 난다. 그 두 양이 3차원 위치로 풀리는지가 검출 결과가 말할 수 있는 범위를 정한다.

기저선을 축으로 표적을 돌리면 $R_b$ 변화가 최대 1.4e-14 m [^3] 다 — 그 방향의 정보량은 SNR 과 관측시간에 무관하게 0 이다. 더 오래 보거나 더 세게 쏘아도 그 축은 풀리지 않는다.

| 형상 | 유효 랭크 (/6) | 위치 RMS 오차 |
|---|---|---|
| 1RX (baseline) | 3 | 57.75 m |
| 2RX | 6 | 0.19 m |
| 1RX + AoA(1deg) | 6 | 0.12 m |
| 1RX + AoA(5deg) | 6 | 0.60 m |

출처 [^15]

표의 랭크는 위치 3 + 속도 3 = **6 차원 상태**에 대한 값이고, 관측창 3 s [^16] 를 누적한 그램행렬에서 센다 — 머리줄의 «랭크 2 [^2]» 는 **한 순간 · 위치 3차원**에 대한 값이라 다른 양이다.

## 이 표의 절대값이 매달린 것

랭크는 셀을 갈아도 그대로지만, 위치 RMS 는 아니다. 표의 절대값은 기준 셀 nr100_G3 [^6] — 위 표의 마지막 줄인 **PRS**, 즉 측위 세션에서만 켜지는 신호에서 푼 값이다. 상시 기준신호 셀은 같은 표에서 $\sigma_{R_b}$ 가 더 크므로 위치 RMS 도 그만큼 커진다.

기하도 함께 매달려 있다 — 기저선 15.07 m [^1] · EIRP 12 dBm [^17] · CPI 0.03 s [^18] 의 챔버 통제 기하다.

![f7_observability](../outputs/figures/report04_f7_observability.png)

**그림 2.** 송수신 한 쌍에 무엇을 더하면 표적 위치가 풀리는가?

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 수신기 2대 형상으로 검출 실험을 재설계한다 | 위치 RMS 0.19 m [^5] 가 검출 실험에서 확인된다 | [편 66 «코히어런트 배열이득은 10log₁₀N 상한에…»](66_rx-elements.ipynb) |
| 도래각 오차를 실제 배열 교정오차로 바꿔 랭크를 다시 센다 | 1RX + AoA 형상이 실제 하드웨어에서 몇 랭크인지 확정된다 | [편 67 «X410 의 12-bit ADC 동적범위가 직…»](67_hardware.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 18개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/verify_observability.json` | `meta.L_m` | 15.07 |
| [^2] | `outputs/verify_observability.json` | `summary.snapshot_fim_rank` | 2 |
| [^3] | `outputs/verify_observability.json` | `summary.exact_rotation_max_dRb_m` | 1.421e-14 |
| [^4] | `outputs/verify_observability.json` | `summary.fix_2rx_rank` | 6 |
| [^5] | `outputs/verify_observability.json` | `summary.fix_2rx_pos_rms_m` | 0.1896 |
| [^6] | `outputs/verify_observability.json` | `gramian.ref_cfg` | nr100_G3 |
| [^7] | `outputs/verify_observability.json` | `cells[2].ref_bw_mhz` | 7.2 |
| [^8] | `outputs/verify_observability.json` | `cells[2].drb_bw_m` | 41.64 |
| [^9] | `outputs/verify_observability.json` | `cells[3].drb_bw_m` | 3.05 |
| [^10] | `outputs/verify_observability.json` | `cells[0].t_cpi` | 0.032 |
| [^11] | `outputs/verify_observability.json` | `cells[0].dfd_hz` | 31.25 |
| [^12] | `outputs/verify_observability.json` | `cells[3].ref_bw_mhz` | 98.28 |
| [^13] | `outputs/verify_observability.json` | `cells` | (4행 표) |
| [^14] | `outputs/verify_observability.json` | `meta.note` | ΔRb = c/B (bistatic; Rb = c·tau, no factor 2). SNR for… |
| [^15] | `outputs/verify_observability.json` | `fixes` | (5항목 묶음) |
| [^16] | `outputs/verify_observability.json` | `gramian.t_obs_s` | 3 |
| [^17] | `outputs/verify_observability.json` | `meta.eirp_dbm` | 12 |
| [^18] | `outputs/verify_observability.json` | `meta.t_cpi_s` | 0.03 |